In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
folder_path = '/content/drive/MyDrive/ML/dataset_dogs/dogs'
if os.path.exists(folder_path):
    print("Files in folder:", os.listdir(folder_path))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D,  MaxPool2D, Dropout, BatchNormalization

import os
from tqdm import tqdm

from sklearn.metrics import accuracy_score

from PIL import Image
import gc

In [ ]:
train_path = "/content/drive/MyDrive/ML/dataset_dogs/dogs/train"
test_path = "/content/drive/MyDrive/ML/dataset_dogs/dogs/test"

In [ ]:
def read_data(path, size=(224, 224)):
    classes = {}

    k = 0
    for p, fol, file in os.walk(path):
        for f in file:
            if f.endswith(".jpg"):
                k += 1

    images = np.zeros((k, *size, 3), dtype=np.float32)
    labels = np.zeros((k,), dtype=np.float32)

    ind = 0
    for i, item_name in enumerate(sorted(os.listdir(path))):
        item_path = os.path.join(path, item_name)
        if os.path.isdir(item_path):
            classes[i] = item_name
            for img_name in tqdm(os.listdir(item_path)):
                if not img_name.endswith(".jpg"):
                    continue
                img_path = os.path.join(item_path, img_name)

                with Image.open(img_path) as img:
                    img = np.asarray(img.convert("RGB").resize(size)) / 255.

                images[ind] = img
                labels[ind] = i
                ind += 1

                if ind % 1000 == 0:
                    gc.collect()
        else:
            continue

    return classes, images, labels

In [ ]:
classes, x_train, y_train = read_data(train_path, size=(224,224))

In [ ]:
classes

In [ ]:
classes, x_test, y_test = read_data(test_path, size=(224, 224))

In [ ]:
classes

In [ ]:
plt.figure(figsize=(25, 30))
for i in range(100):
    plt.subplot(10, 10, i + 1)
    ind = np.random.randint(0, len(x_train) - 1)
    plt.imshow(x_train[ind])
    plt.xticks([])
    plt.yticks([])
    plt.title(classes[y_train[ind]])

In [ ]:
y_train_cat = tf.keras.utils.to_categorical(y_train)
y_test_cat = tf.keras.utils.to_categorical(y_test)

In [ ]:
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Model

densenet_base = DenseNet201(weights='imagenet', include_top=False, input_shape=x_train.shape[1:])

for layer in densenet_base.layers:
    layer.trainable = False

x = densenet_base.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(len(classes), activation='softmax')(x)

model = Model(inputs=densenet_base.input, outputs=predictions)

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=15, batch_size=32)